# 条件付き敵対的生成ネットワーク（cGAN: Conditional Generative Adversarial Networks）

**学習目標**

- Keras で条件付き GAN（cGAN）アーキテクチャ（Generator と Discriminator で構成）を構築する
- Generator と Discriminator の損失関数を定義する
- keras.Model のサブクラス化を使用して GAN のトレーニングステップを定義する
- MNIST データセットで cGAN をトレーニングする

## はじめに
このノートブックでは、[条件付き敵対的生成ネットワーク（Conditional Generative Adversarial Network）](https://arxiv.org/abs/1411.1784)を構築してトレーニングする方法を学びます。

前のラボ（[deepconv-gan.ipynb](./deepconv_gan.ipynb)）で説明した基本的な GAN は、生成方法を学習したい画像を対象に、完全に教師なしの方法でトレーニングされます。<br>
その後、ランダムなノイズベクトルのような潜在表現を使用して、学習された画像空間を探索しサンプリングします。

簡単な機能強化として、入力にラベル付きの外部フラグを追加することが挙げられます。<br>
例えば、0 から 9 までの手書き数字で構成される MNIST データセットを考えてみましょう。

通常、GAN は数字の分布のみを学習し、Generator にランダムなノイズベクトルが与えられると、さまざまな数字を生成します。

![基本的な GAN](https://user-images.githubusercontent.com/6895245/171208971-3ff1c023-cdd7-47c2-b734-77e07b8eb6ac.png)

しかし、どの数字が生成されるかを制御することはできません。

MNIST のようにトレーニング中、各画像の実際の数字やクラスの指定がわかっている場合があります。<br>
その追加情報を GAN トレーニングの特徴量として含めることで、推論時に使用できるようになります。

条件付き GAN（cGAN）を使用すると、画像生成を数字で条件付けることができるため、関心のある特定の数字の分布に焦点を絞ることができます。

これにより、推論時にランダムな数字を受け取るのではなく、希望する数字を渡すことで、特定の数字の画像を生成できるようになります。

![cGAN](https://user-images.githubusercontent.com/6895245/171209451-4b95c856-fafa-40b4-87b6-c7232c1de841.png)

このノートブックでは、条件付き GAN の構築方法を学びます。

## ライブラリのインポート

In [ ]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

In [ ]:
import time

import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from keras.layers import (
    BatchNormalization,
    Concatenate,
    Conv2D,
    Conv2DTranspose,
    Dense,
    Embedding,
    Flatten,
    Input,
    LeakyReLU,
    Reshape,
)

## 定数

In [ ]:
IMAGE_SHAPE = (28, 28, 1)
NUM_CLASSES = 10
LATENT_DIM = 512
PROJECTION_SHAPE = (7, 7, 56)

## 条件付き GAN 用の Generator および Discriminator モデルの作成

前のラボでは、通常の GAN は [Keras Sequential API](https://keras.io/api/models/sequential/) を使用して定義されましたが、ここでは [Keras Functional API](https://keras.io/guides/functional_api/) を使用して、条件付き GAN 用のより複雑なモデルを作成します。

まず、Generator と Discriminator の両方で使用される、数字（0〜9）の情報を埋め込むレイヤーを定義する必要があります。<br>
後で MNIST 画像と結合できるように、埋め込みは画像のような階数 3 のテンソル（高さ、幅、1）にする必要があります。

In [ ]:
def create_digit_embedding(digits, num_classes, embedding_dim, image_shape):
    embedded_digits = Embedding(
        name="digit_embedding", input_dim=num_classes, output_dim=embedding_dim
    )(digits)

    num_pixels = image_shape[0] * image_shape[1]  # 高さ * 幅

    dense_digits = Dense(name="digit_dense", units=num_pixels)(embedded_digits)

    digits_image = Reshape(
        name="digit_to_image_reshape",
        target_shape=(image_shape[0], image_shape[1], 1),
    )(dense_digits)

    return digits_image

数字の埋め込み画像を生成してみましょう。まだトレーニングされていないため、ランダムな画像にすぎません。

In [ ]:
DIGIT_TO_EMBED = 0  # [0,9]

digit_embedding = create_digit_embedding(
    keras.ops.convert_to_tensor([DIGIT_TO_EMBED]), NUM_CLASSES, 128, (7, 7, 1)
)[0]
plt.imshow(digit_embedding)

### Generator

条件付き GAN の Generator は、ランダムな潜在ベクトルと数字ベクトルから画像を生成します。
`latent_dim` の形状を持つノイズベクトルと数字のインデックスを受け取ります。両方のベクトルは埋め込まれ、結合するために `projection_shape` で定義された同じ高さと幅に変形されます。次に Generator は、結合された階数 3 のベクトルから 28x28x1 の MNIST 画像を生成します。

Generator モデルは、通常 [BatchNormalization](https://keras.io/api/layers/normalization_layers/batch_normalization/) とともに [Conv2DTranspose](https://keras.io/api/layers/initializers/) を使用して画像をアップサンプリングします。また、非線形活性化関数として [LeakyRelu](https://keras.io/api/layers/activation_layers/leaky_relu/) を使用します。

次に、cGAN の Generator を構築するための関数 `create_dcgan_generator` を定義します。これは以下を行います。

1. 潜在ベクトルを階数 3 のテンソルにマッピングします。ここでは `projection_shape` 変数を使用して潜在画像の形状を定義します。
2. 数字のインデックスを階数 3 のテンソルにマッピングします（`create_digit_embedding` 関数を呼び出します）。出力テンソルの形状は、次のステップで画像と結合できるように、`projection_shape` と同じ高さと幅にフォーマットされます。
3. 1 と 2 を結合（concatenate）します。
4. 3 を、`Conv2DTranspose`、`BatchNormalization`、`LeakyRelu` レイヤーで構成されるブロックを持つ GAN Generator に渡します。この部分は前のノートブックの DCGAN Generator とほぼ同じであることに注意してください。
5. Model オブジェクトを返します。

In [ ]:
def create_dcgan_generator(latent_dim, num_classes, projection_shape):
    # ランダムベクトルの入力
    latent_vector = Input(name="latent_input", shape=(latent_dim,))

    num_image_pixel = (
        projection_shape[0] * projection_shape[1] * projection_shape[2]
    )

    x = Dense(name="latent_dense", units=num_image_pixel)(latent_vector)
    x = LeakyReLU(name="latent_leakyrelu", alpha=0.2)(x)

    latent_image = Reshape(
        name="latent_to_image_reshape", target_shape=projection_shape
    )(x)

    # 数字の入力を受け取る（[0, 9]）
    digits = Input(name="digit_input", shape=())
    # 数字を階数 3 のテンソルである数字埋め込みに射影する
    digit_embeddings = create_digit_embedding(
        digits=digits,
        num_classes=num_classes,
        embedding_dim=50,
        image_shape=projection_shape,
    )

    concatenated_inputs = Concatenate(name="concatenate_latent_digit", axis=-1)(
        inputs=[latent_image, digit_embeddings]
    )

    x = Conv2DTranspose(128, 4, 2, "same")(concatenated_inputs)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)

    x = Conv2DTranspose(128, 4, 2, "same")(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)

    outputs = Conv2DTranspose(
        1, 3, 1, "same", activation="tanh", name="generator_output"
    )(x)

    return keras.Model(
        inputs=[latent_vector, digits], outputs=outputs, name="dcgan_generator"
    )

In [ ]:
dcgan_generator = create_dcgan_generator(
    latent_dim=LATENT_DIM,
    num_classes=NUM_CLASSES,
    projection_shape=PROJECTION_SHAPE,
)
dcgan_generator.summary()

Generator モデルをプロットして、どのように構築されているか確認してみましょう。

In [ ]:
# Generator モデルのプロット。
keras.utils.plot_model(
    model=dcgan_generator, show_shapes=True, show_layer_names=True, dpi=75
)

### Discriminator

次に、Discriminator を構築します。Discriminator は CNN ベースの画像分類器です。<br>
28x28x1 の形状の画像と、同様に 28x28x1 に埋め込まれ変形された数字インデックスを受け取り、それらを結合して、その画像が本物であるかどうかを示す単一の分類結果を返します。

cGAN の Discriminator を構築するための関数 `create_dcgan_discriminator` を定義しましょう。これは以下を行います。

1. `image_shape` の形状を持つ画像入力（本物または偽物のいずれか）を受け取ります。
2. 数字のインデックスを階数 3 のテンソルにマッピングします（`create_digit_embedding` 関数を呼び出します）。出力テンソルの形状は、次のステップで画像と結合できるように、`image_shape` と同じ高さと幅にフォーマットされます。
3. 1 と 2 を結合（concatenate）します。
4. 3 を、`Conv2D`、`BatchNormalization`、`LeakyRelu` レイヤーで構成されるブロックを持つ GAN Discriminator に渡します。この部分は前のノートブックの DCGAN Discriminator とほぼ同じであることに注意してください。
5. Model オブジェクトを返します。

In [ ]:
def create_dcgan_discriminator(image_shape, num_classes):
    images = Input(name="image_input", shape=image_shape)

    digits = Input(name="digit_input", shape=())
    digit_embeddings = create_digit_embedding(
        digits=digits,
        num_classes=num_classes,
        embedding_dim=50,
        image_shape=image_shape,
    )

    concatenated_inputs = Concatenate(name="concatenate_image_digit", axis=-1)(
        [images, digit_embeddings]
    )

    x = Conv2D(64, 3, 1, "same")(concatenated_inputs)

    x = Conv2D(128, 3, 2, "same")(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)

    x = Conv2D(128, 3, 2, "same")(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)

    x = Conv2D(256, 3, 2, "same")(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)

    x = Flatten()(x)
    outputs = Dense(1, name="discriminator_output")(x)

    return keras.Model(
        inputs=[images, digits], outputs=outputs, name="dcgan_discriminator"
    )

In [ ]:
dcgan_discriminator = create_dcgan_discriminator(
    image_shape=IMAGE_SHAPE, num_classes=NUM_CLASSES
)
dcgan_discriminator.summary()

In [ ]:
# Discriminator モデルのプロット。
keras.utils.plot_model(
    model=dcgan_discriminator, show_shapes=True, show_layer_names=True, dpi=75
)

Generator と Discriminator ネットワークのトレーニングを容易にするために使用できる、`GAN` という名前のカスタム `Model` クラスを作成します。

## トレーニングループの定義

GAN のトレーニングでは、Discriminator と Generator の最適化目標が異なるため、これら 2 つのモデルを「交互に」トレーニングします。<br>
この場合、モデルを交互にトレーニングするとは、Discriminator をトレーニングするときは Generator のパラメータが凍結されて更新されないことを意味します。その後、Discriminator を凍結した状態で Generator をトレーニングします。


ここでは、2 つのモデルに対して個別のトレーニングステップを定義しましょう。

### Discriminator のトレーニングステップ

以下の `discriminator_train_step` 関数では、以下のステップを含む Discriminator 固有のトレーニングルーチンを定義します。
1. 潜在空間からランダムな点をサンプリングする。
2. ステップ 1 で作成されたランダムベクトルからのノイズから画像を生成する。
3. 生成された画像と本物の画像を結合して、トレーニング特徴量を作成する。
4. 生成された画像用のラベル（0）と本物の画像用のラベル（1）を作成する。ここでは、本物のラベルに**片側ラベル平滑化（one-sided label smoothing）**を適用します。片側ラベル平滑化の説明については以下を参照してください。
5. `tf.GradientTape` コンテキスト内で Discriminator を呼び出して予測を取得し、損失を計算する。
6. `GradientTape` から Discriminator のパラメータの勾配を取得し、それらを更新する。Discriminator のトレーニングステップでは Generator は更新されないことに注意してください。

**片側ラベル平滑化（one-sided label smoothing）**: 本物のラベル（1）に [0.0, 1.0] の範囲の浮動小数点定数を掛けることで、Discriminator が画像内のごく一部の特徴のみに基づいて過度に自信を持って予測するのを防ぎます。これにより、Generator がその弱点を悪用して Discriminator を打ち負かすことだけに特化し、画像生成自体が上手くならなくなる事態を防ぎます。

In [ ]:
def discriminator_train_step(
    discriminator,
    generator,
    discriminator_optimizer,
    loss_fn,
    discriminator_accuracy,
    one_sided_label_smoothing,
    batch_size,
    real_images,
    input_digits,
):
    # 潜在空間からランダムな点をサンプリング。
    random_latent_vectors = keras.random.normal(shape=(batch_size, LATENT_DIM))

    # ノイズから画像を生成。
    generated_images = generator(inputs=[random_latent_vectors, input_digits])

    # 生成された画像と本物の画像を結合。
    combined_images = keras.ops.concatenate(
        [generated_images, real_images], axis=0
    )

    # 偽物と本物のラベルを作成。
    fake_labels = keras.ops.zeros(shape=(batch_size, 1))
    real_labels = keras.ops.ones(shape=(batch_size, 1))

    # トレーニングを支援するために本物のラベルを平滑化。
    real_labels *= one_sided_label_smoothing

    # 結合された画像に合わせてラベルを結合。
    labels = keras.ops.concatenate([fake_labels, real_labels], axis=0)

    # Discriminator のみをトレーニング。
    with tf.GradientTape() as tape:
        predictions = discriminator(
            inputs=[
                combined_images,
                keras.ops.concatenate([input_digits, input_digits], axis=0),
            ]
        )
        discriminator_loss = loss_fn(y_true=labels, y_pred=predictions)

    pred_prob = keras.ops.sigmoid(predictions)
    discriminator_accuracy.update_state(y_true=labels, y_pred=pred_prob)

    grads = tape.gradient(
        target=discriminator_loss,
        sources=discriminator.trainable_weights,
    )

    discriminator_optimizer.apply_gradients(
        grads_and_vars=zip(grads, discriminator.trainable_weights)
    )

    return discriminator_loss, discriminator_accuracy.result()

### Generator のトレーニングステップ

同様に、`generator_train_step` 関数で Generator 固有のトレーニングステップを定義します。これには以下のステップが含まれます。
1. 潜在空間からランダムな点をサンプリングする。
2. 生成された画像に対して、あたかも本物の画像であるかのようにラベル（1）を作成する。
3. `tf.GradientTape` コンテキスト内で Generator を呼び出して画像を生成し、それらを Discriminator に渡して、ステップ 2 で作成したラベルを使用して損失を計算する。
4. `GradientTape` から Generator のパラメータの勾配を取得し、それらを更新する。Generator のトレーニングステップでは Discriminator は更新されないことに注意してください。

In [ ]:
def generator_train_step(
    discriminator,
    generator,
    generator_optimizer,
    loss_fn,
    batch_size,
    input_digits,
):
    # 潜在空間からランダムな点をサンプリング。
    random_latent_vectors = keras.random.normal(shape=(batch_size, LATENT_DIM))

    # あたかも本物の画像であるかのようにラベルを作成。
    labels = keras.ops.ones(shape=(batch_size, 1))

    # Generator のみをトレーニング。
    with tf.GradientTape() as tape:
        fake_images = generator(inputs=[random_latent_vectors, input_digits])
        predictions = discriminator(inputs=[fake_images, input_digits])
        generator_loss = loss_fn(y_true=labels, y_pred=predictions)

    grads = tape.gradient(
        target=generator_loss, sources=generator.trainable_weights
    )

    generator_optimizer.apply_gradients(
        grads_and_vars=zip(grads, generator.trainable_weights)
    )

    return generator_loss

### 外部トレーニングループ
これで両方のトレーニングループが定義されました。<br>
それらを `train_step` 関数でラップし、外部トレーニングループを定義しましょう。

ここでは、Discriminator と Generator を順次トレーニングするために `discriminator_train_step` と `generator_train_step` を呼び出し、2 つのモデルの 2 つの損失を返します。

In [ ]:
DISCRIMINATOR_OPTIMIZER = keras.optimizers.Adam(
    learning_rate=0.0002, beta_1=0.5, beta_2=0.999, epsilon=1e-8
)
GENERATOR_OPTIMIZER = keras.optimizers.Adam(
    learning_rate=0.0004, beta_1=0.5, beta_2=0.999, epsilon=1e-8
)

LOSS_FN = keras.losses.BinaryCrossentropy(from_logits=True)

ONE_SIDED_LABEL_SMOOTHING = 0.9

DISCRIMINATOR_ACCURACY = keras.metrics.BinaryAccuracy()


def train_step(features):
    real_images = features["images"]
    input_digits = features["digits"]

    # ランダムノイズベクトルに使用する動的なバッチサイズを取得。
    batch_size = keras.ops.shape(real_images)[0]

    # まず Discriminator をトレーニング。
    discriminator_loss, discriminator_accuracy_value = discriminator_train_step(
        dcgan_discriminator,
        dcgan_generator,
        DISCRIMINATOR_OPTIMIZER,
        LOSS_FN,
        DISCRIMINATOR_ACCURACY,
        ONE_SIDED_LABEL_SMOOTHING,
        batch_size,
        real_images,
        input_digits,
    )

    # 次に Generator をトレーニング。
    generator_loss = generator_train_step(
        dcgan_discriminator,
        dcgan_generator,
        GENERATOR_OPTIMIZER,
        LOSS_FN,
        batch_size,
        input_digits,
    )
    losses = {
        "discriminator_loss": discriminator_loss,
        "discriminator_accuracy": discriminator_accuracy_value,
        "generator_loss": generator_loss,
    }

    return losses

### tf.data による入力データパイプラインの作成
これでモデルの準備が整いました。tf.data を使用してデータパイプラインを作成しましょう。<br>
cGAN には画像だけでなくラベルも渡す必要があることに注意してください。

In [ ]:
def create_train_dataset(batch_size):
    # MNIST データの取得と整形。
    (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

    # 訓練セットとテストセットの両方を結合。
    combined_images = np.concatenate([x_train, x_test])
    combined_images = (combined_images.astype("float32") - 127.5) / 255.0
    combined_images = np.reshape(combined_images, newshape=(-1, 28, 28, 1))

    combined_labels = np.concatenate([y_train, y_test])

    # 訓練用の tf.data.Dataset を作成。
    dataset = tf.data.Dataset.from_tensor_slices(
        tensors={"images": combined_images, "digits": combined_labels}
    )
    dataset = dataset.shuffle(buffer_size=10000).batch(
        batch_size=batch_size, drop_remainder=True
    )

    return dataset

In [ ]:
BATCH_SIZE = 128
EPOCHS = 10  # より良い生成結果を得るには数値を変更してください

training_dataset = create_train_dataset(batch_size=BATCH_SIZE)

dataset_length = training_dataset.cardinality().numpy()

## cGAN のトレーニング
ループ内で `train_step` 関数を呼び出すことで、モデルをトレーニングできます。<br>
GPU（NVIDIA T4）を使用すると、10 エポックのトレーニングに約 7 分かかります。

In [ ]:
output_keys = ["discriminator_loss", "discriminator_accuracy", "generator_loss"]

history = {key: [] for key in output_keys}

for epoch in range(EPOCHS):
    print(f"{'=' * 20} EPOCH {epoch + 1} {'=' * 20}")
    epoch_history = {key: [] for key in output_keys}

    for i, features in enumerate(training_dataset, 1):
        losses = train_step(features)
        for key in output_keys:
            epoch_history[key].append(losses[key])

        if i % 100 == 0 or i == dataset_length:
            performance_str = ", ".join(
                [
                    f"{key}: {keras.ops.mean(epoch_history[key]):5f}"
                    for key in output_keys
                ]
            )
            print(f"step {i:3d} / {dataset_length}: {performance_str}")

    for key in output_keys:
        history[key].append(keras.ops.mean(epoch_history[key]))

    DISCRIMINATOR_ACCURACY.reset_state()

トレーニング済みモデルを使用して、損失の推移と生成されたいくつかの画像をプロットしてみましょう。

In [ ]:
def plot_loss_history(history):
    plt.plot(history["discriminator_loss"])
    plt.plot(history["discriminator_accuracy"])
    plt.plot(history["generator_loss"])
    plt.title("Training losses")
    plt.ylabel("Loss")
    plt.xlabel("Epoch")
    plt.legend(output_keys, loc="upper left")
    plt.show()

In [ ]:
plot_loss_history(history=history)

損失曲線はどのように見えますか？

Discriminator と Generator は異なる損失値を最適化しようと互いに競い合っていることを思い出してください。<br>
したがって、トレーニング中に一方の損失が減少し、もう一方が急激に増加している場合は、2 つのモデル間のバランスに問題があることがわかります。その場合は、最適化戦略、学習率スケジューラー、またはモデルアーキテクチャ自体を調整して、これら 2 つのバランスをとる必要があります。

また、Discriminator の精度（accuracy）を監視することも有用です。<br>
50% に近い場合、Discriminator は本物か偽物かを分類できなくなっていることを意味します。これにより、Generator が非常に優れたリアルな画像を生成しているか、あるいは Discriminator がまったく適合していないかのいずれかであることが理解できます。

## 画像の生成
これで、cGAN で画像を生成する準備が整いました。Generator にどの数字を生成させたいかを指定できることに注意してください。

In [ ]:
LABEL_TO_GENERATE = 9  # 0-9
latent_vectors = keras.random.normal(shape=(1, LATENT_DIM))

generated_image = dcgan_generator(
    inputs=[
        latent_vectors,
        keras.ops.convert_to_tensor([LABEL_TO_GENERATE], dtype="int32"),
    ]
)
plt.imshow(
    keras.ops.reshape(generated_image, generated_image.shape[1:-1]), cmap="gray"
)

複数の数字を生成してみましょう。

これは、複数の画像を可視化するためのヘルパー関数です。

In [ ]:
def plot_images(images):
    """画像をプロットします。

    Args:
        images: np.array, array of images of
            [num_images, image_size, image_size, num_channels].
    """
    num_images = len(images)

    plt.figure(figsize=(20, 20))
    for i in range(num_images):
        image = images[i]
        plt.subplot(1, num_images, i + 1)
        plt.xticks([])
        plt.yticks([])
        plt.grid(False)
        plt.imshow(keras.ops.reshape(image, image.shape[:-1]), cmap="gray")
    plt.show()

In [ ]:
num_examples = 20
latent_vectors = keras.random.normal(shape=(num_examples, LATENT_DIM))
for i in range(10):
    labels = keras.ops.ones(shape=num_examples, dtype="int32") * i
    plot_images(images=dcgan_generator(inputs=[latent_vectors, labels]))

いい感じです！エポック数を大きく（50 など）指定することで、より良い画像を取得してみることができます。

## （オプション）GAN クラスの定義
分離されたトレーニングループを定義する代わりに、[keras.Model](https://keras.io/api/models/model/) をサブクラス化して、それらを 1 つのクラスにまとめることもできます。

独自のモデルクラスを定義することで、元のトレーニングステップを簡単に定義し、`model.fit()` などの標準的なモデル API を使用できるようになります。また、keras.Model の関数をオーバーライドして、モデルの動作を変更することもできます。

上記で定義されたトレーニング関数に加えて、ここでは Conditional_GAN クラスに以下の関数を追加します。
- `__init__()`: モデルクラスをインスタンス化します。ここでは、上記で定義した Generator と Discriminator を渡します。また、Discriminator のトレーニングで使用される `one_sided_label_smoothing` 定数もここで指定します。
- `compile()`: `Model.compile()` 関数をオーバーライドして、Generator と Discriminator のオプティマイザを個別に定義し、両方のモデルで使用される損失関数を追加できるようにします。
- `metrics()`: モデルの評価指標を定義します。この `metrics` プロパティにいくつかの指標を追加することで、各エポックの開始時や `evaluate()` の開始時に `reset_states()` が自動的に呼び出されるようになります。

In [ ]:
class Conditional_GAN(keras.Model):
    def __init__(
        self, discriminator, generator, latent_dim, one_sided_label_smoothing
    ):
        super().__init__()
        self.discriminator = discriminator
        self.generator = generator
        self.latent_dim = latent_dim
        self.one_sided_label_smoothing = one_sided_label_smoothing

    def compile(
        self,
        discriminator_optimizer,
        generator_optimizer,
        loss_fn,
        discriminator_accuracy,
    ):
        super().compile()
        self.discriminator_optimizer = discriminator_optimizer
        self.generator_optimizer = generator_optimizer
        self.loss_fn = loss_fn
        self.discriminator_accuracy = discriminator_accuracy

    def discriminator_train_step(self, batch_size, real_images, input_digits):
        # 上記で定義した関数を再利用
        return discriminator_train_step(
            self.discriminator,
            self.generator,
            self.discriminator_optimizer,
            self.loss_fn,
            self.discriminator_accuracy,
            self.one_sided_label_smoothing,
            batch_size,
            real_images,
            input_digits,
        )

    def generator_train_step(self, batch_size, input_digits):
        # 上記で定義した関数を再利用
        return generator_train_step(
            self.discriminator,
            self.generator,
            self.generator_optimizer,
            self.loss_fn,
            batch_size,
            input_digits,
        )

    def train_step(self, features):
        real_images = features["images"]
        input_digits = features["digits"]

        # ランダムノイズベクトルに使用する動的なバッチサイズを取得。
        batch_size = keras.ops.shape(real_images)[0]

        # まず Discriminator をトレーニング。
        (
            discriminator_loss,
            discriminator_accuracy_value,
        ) = self.discriminator_train_step(batch_size, real_images, input_digits)

        # 次に Generator をトレーニング。
        generator_loss = self.generator_train_step(batch_size, input_digits)
        losses = {
            "discriminator_loss": discriminator_loss,
            "discriminator_accuracy": discriminator_accuracy_value,
            "generator_loss": generator_loss,
        }
        return losses

    @property
    def metrics(self):
        return [self.discriminator_accuracy]

Conditional GAN モデルをインスタンス化してコンパイルできます。

インスタンス化によって `__init__` 関数が呼び出され、コンパイルによって `Conditional_GAN` クラスの `compile` 関数が呼び出されることに注意してください。

In [ ]:
dcgan_generator = create_dcgan_generator(
    latent_dim=LATENT_DIM,
    num_classes=NUM_CLASSES,
    projection_shape=PROJECTION_SHAPE,
)

dcgan_discriminator = create_dcgan_discriminator(
    image_shape=IMAGE_SHAPE, num_classes=NUM_CLASSES
)

# DCGAN の Generator と Discriminator を使用して Conditional_GAN インスタンスを作成。
dcgan = Conditional_GAN(
    discriminator=dcgan_discriminator,
    generator=dcgan_generator,
    latent_dim=LATENT_DIM,
    one_sided_label_smoothing=0.9,
)

dcgan.compile(
    discriminator_optimizer=keras.optimizers.Adam(
        learning_rate=0.0002, beta_1=0.5, beta_2=0.999, epsilon=1e-8
    ),
    generator_optimizer=keras.optimizers.Adam(
        learning_rate=0.0004, beta_1=0.5, beta_2=0.999, epsilon=1e-8
    ),
    loss_fn=keras.losses.BinaryCrossentropy(from_logits=True),
    discriminator_accuracy=keras.metrics.BinaryAccuracy(),
)

## cGAN のトレーニング
単に `model.fit()` を呼び出すだけでモデルをトレーニングできます。<br>
GPU（NVIDIA T4）を使用すると、10 エポックのトレーニングに約 7 分かかります。

In [ ]:
BATCH_SIZE = 128
EPOCHS = 10  # より良い生成結果を得るには数値を変更してください

# DCGAN モデルのトレーニング。
dcgan_history = dcgan.fit(
    create_train_dataset(batch_size=BATCH_SIZE), epochs=EPOCHS
)

In [ ]:
plot_loss_history(history=dcgan_history.history)

In [ ]:
num_examples = 20
latent_vectors = keras.random.normal(shape=(num_examples, LATENT_DIM))
for i in range(10):
    labels = keras.ops.ones(shape=num_examples, dtype="int32") * i
    plot_images(images=dcgan_generator(inputs=[latent_vectors, labels]))


## まとめ
このノートブックでは、条件付き GAN（Conditional GAN）モデルを構築してトレーニングする方法を学びました。

Copyright 2022 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License